# Simulation 1 Start-Drift Sweep Cued Retrieval Figure

> Generate a parallel Figure 2 candidate that varies retrieval start drift while preserving the current unguided/VIT-style cue setup.

This notebook crosses reminder condition, task encoding strength, and retrieval start drift.
It keeps the supplied film cue before recall constant, leaves film-source orientation off, and saves topline film-recall bars plus supporting CSV summaries.

In [ ]:
import csv
import os
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown
from jax import random, tree_util, vmap
from jaxcmr.analyses.pnr import fixed_pres_pnr
from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root

from selective_interference_v2 import (
    PHASE_COLORS,
    Paradigm,
    add_phase_bands,
    apply_supplied_item_context_cue,
    batch_trial,
    configure_rates,
    light_to_dark_colors,
    load_fit_params,
    make_factory,
    make_is_emotional,
    make_is_target,
    prepare_sweep,
    save_figure,
    simulate_periodic_cued_free_recall,
    split_scales_for_cache,
    standard_remap,
    sweep_rngs,
)

warnings.filterwarnings("ignore")

In [ ]:
# --- sweep configuration ---
PROJECT_ROOT = ""
FIT_PATH = "results/fits/Dupertuys2026_eCMR_source_only_phi_no_shared_support_best_of_1.json"
FIGURE_DIR = "figures"
FIGURE_STR = "simulation1_start_drift_sweep_cued"
RNG_SEED = 0

TASK_MCF_VALUES = [1.0, 2.0]
TASK_MCF_LABELS = {
    1.0: "Weak encoding",
    2.0: "Strong encoding",
}

PRIMARY_SOURCE_ORIENTATION = 0.0

START_DRIFT_BASE_VALUES = [0.0, 1.0]
START_DRIFT_CANDIDATES = [1.5, 2.0, 2.5]
START_DRIFT_LABELS = {
    0.0: "Start drift 0.0",
    1.0: "Start drift 1.0",
    1.5: "Start drift 1.5",
    2.0: "Start drift 2.0",
    2.5: "Start drift 2.5",
}

FILM_CUE_REINSTATEMENT = 0.90
CUE_INTERVAL = 4
FIRST_CUE_AFTER = 0

N_FILM = 16
N_BREAK = 16
N_INTERFERENCE = 16
N_FILLER = 16
EXPERIMENT_COUNT = 5000
MAX_RECALL = 48

FILM_EMOTIONAL = True
INTERFERENCE_EMOTIONAL = False
CACHE_AFTER = "filler"

EXTRA_FIXED_SCALES = {
    "source_learning_baseline": 0.05,
    "primacy_scale": 0.1,
    "primacy_decay": 1.5,
}

BASE_FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

REMINDER_CONDITIONS = {
    "No reminder": {
        "reminder_start_drift_scale": 0.0,
        "reminder_drift_scale": 0.0,
    },
    "With reminder": {
        "reminder_start_drift_scale": 1.0,
        "reminder_drift_scale": 1.0,
    },
}

RETRIEVAL_FIXED_SCALES = {
    "tau_scale": 1.0,
    "target_recall_drift_scale": 1.0,
    "rejected_recall_drift_scale": 1.0,
    "recall_drift_scale": 1.0,
}

In [ ]:
#| output: asis
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    project_root = Path(find_project_root())

fit_path = project_root / FIT_PATH
figure_dir = project_root / FIGURE_DIR if FIGURE_DIR else None
task_mcf_values = np.asarray(TASK_MCF_VALUES, dtype=float)
start_drift_diagnostic_values = np.asarray(
    START_DRIFT_BASE_VALUES + START_DRIFT_CANDIDATES,
    dtype=float,
)
extra_fixed_scales = {**EXTRA_FIXED_SCALES}
base_fixed_scales = {**BASE_FIXED_SCALES, **extra_fixed_scales}

params, n_subjects = load_fit_params(fit_path)
paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    experiment_count=EXPERIMENT_COUNT,
    max_recall=MAX_RECALL,
)
factory = make_factory(
    is_emotional=make_is_emotional(
        paradigm,
        film_emotional=FILM_EMOTIONAL,
        interference_emotional=INTERFERENCE_EMOTIONAL,
    ),
    is_target=make_is_target(paradigm),
)
rng = random.PRNGKey(RNG_SEED)

Markdown(
    f"Loaded {n_subjects} fit parameter set(s). Running {len(REMINDER_CONDITIONS)} reminder conditions, "
    f"{len(task_mcf_values)} task encoding values, and {len(start_drift_diagnostic_values)} start-drift values "
    f"with a film cue before first recall and {EXPERIMENT_COUNT} simulated trials per cell."
)

In [ ]:
def position_phase_labels(paradigm):
    return (
        ["film"] * paradigm.n_film
        + ["break"] * paradigm.n_break
        + ["task"] * paradigm.n_interference
        + ["filler"] * paradigm.n_filler
    )


def write_rows(path, fieldnames, rows):
    os.makedirs(path.parent, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def phase_recall_totals(spc):
    totals = {}
    for phase in sorted(set(position_phase)):
        positions = [idx for idx, value in enumerate(position_phase) if value == phase]
        totals[phase] = float(np.sum(spc[positions]))
    return totals


def encode_items(model, items, method):
    for item in np.asarray(items, dtype=int):
        model = getattr(model, method)(jnp.asarray(int(item)))
    return model


def scalar_condition_model(start_drift_scale, reminder_condition, task_scale):
    one_subject_params = {key: value[:1] for key, value in params.items()}
    models = vmap(
        lambda p: factory(paradigm.list_length, p, None),
    )(one_subject_params)
    models = models.replace(n_film=jnp.asarray([paradigm.n_film]))
    fixed_scales = {
        **base_fixed_scales,
        **REMINDER_CONDITIONS[reminder_condition],
        "interference_mcf_scale": float(task_scale),
    }
    configured = configure_rates(
        models,
        **fixed_scales,
        **RETRIEVAL_FIXED_SCALES,
        start_drift_scale=float(start_drift_scale),
        film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
    )
    model = tree_util.tree_map(
        lambda x: x[0]
        if hasattr(x, "shape") and len(x.shape) > 0 and x.shape[0] == 1
        else x,
        configured,
    )
    model = encode_items(model, paradigm.film_items, "experience_film")
    model = encode_items(model, paradigm.break_items, "experience_break")
    model = model.start_reminders()
    model = encode_items(model, paradigm.film_items, "remind")
    model = encode_items(model, paradigm.interference_items, "experience_interference")
    model = encode_items(model, paradigm.filler_items, "experience_filler")
    return model


def support_after_initial_film_cue(start_drift_scale, reminder_condition, task_scale):
    model = scalar_condition_model(start_drift_scale, reminder_condition, task_scale)
    model = model.start_retrieving()
    support_by_cue = []
    for cue_item in np.asarray(paradigm.film_items, dtype=int):
        cued_model = apply_supplied_item_context_cue(
            model,
            jnp.asarray(int(cue_item)),
            FILM_CUE_REINSTATEMENT,
        )
        temporal_support = np.asarray(cued_model.mcf.probe(cued_model.context.state), dtype=float)
        source_support = np.asarray(
            cued_model.emotion_mcf.probe(cued_model.emotion_context.state),
            dtype=float,
        )
        support_by_cue.append((temporal_support + source_support)[:n_presented])
    support = np.mean(np.asarray(support_by_cue), axis=0)
    denom = float(np.sum(support))
    share = support / denom if denom > 0 else np.full_like(support, np.nan)
    return support, share


position_phase = position_phase_labels(paradigm)
n_presented = len(position_phase)

In [ ]:
def cued_recall_trial(
    model,
    rng,
    film_items,
    cue_scale,
    max_recall,
    cue_interval,
    first_cue_after,
):
    recall_rng, initial_cue_rng = random.split(rng)
    model = model.start_retrieving()
    if first_cue_after == 0:
        cue_item = random.choice(initial_cue_rng, film_items)
        model = apply_supplied_item_context_cue(model, cue_item, cue_scale)
        scheduled_first_cue_after = cue_interval
    else:
        scheduled_first_cue_after = first_cue_after
    _, recalls, cue_attempts, cue_items = simulate_periodic_cued_free_recall(
        model,
        max_recall,
        recall_rng,
        film_items,
        cue_scale,
        cue_interval=cue_interval,
        first_cue_after=scheduled_first_cue_after,
    )
    return recalls, cue_attempts, cue_items


batched_cued_recall = batch_trial(cued_recall_trial, n_args=7, n_static=3)

prepared_cache = {}

for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
    for task_scale in task_mcf_values:
        fixed_scales = {
            **base_fixed_scales,
            **reminder_scales,
            "interference_mcf_scale": float(task_scale),
        }
        pre_cache_scales, post_cache_scales = split_scales_for_cache(
            fixed_scales,
            CACHE_AFTER,
        )
        prepared = prepare_sweep(
            params,
            paradigm,
            factory,
            cache_after=CACHE_AFTER,
            **pre_cache_scales,
        )
        prepared_cache[(reminder_condition, float(task_scale))] = prepared

In [ ]:
cue_rows = []
spc_rows = []
pfr_rows = []
cue_curves = {}
pfr_curves = {}
cue_rng = random.PRNGKey(RNG_SEED + 1)

for start_drift_scale in start_drift_diagnostic_values:
    for reminder_condition, reminder_scales in REMINDER_CONDITIONS.items():
        for task_scale in task_mcf_values:
            fixed_scales = {
                **base_fixed_scales,
                **reminder_scales,
                "interference_mcf_scale": float(task_scale),
            }
            _, post_cache_scales = split_scales_for_cache(
                fixed_scales,
                CACHE_AFTER,
            )
            prepared = prepared_cache[(reminder_condition, float(task_scale))]
            ready_models = configure_rates(
                prepared.models,
                **post_cache_scales,
                **RETRIEVAL_FIXED_SCALES,
                start_drift_scale=float(start_drift_scale),
                film_source_start_drift_rate=float(PRIMARY_SOURCE_ORIENTATION),
            )
            rngs, cue_rng = sweep_rngs(
                cue_rng,
                prepared.n_subjects,
                prepared.experiment_count,
            )
            recalls, cue_attempts, cue_items = batched_cued_recall(
                ready_models,
                rngs,
                paradigm.film_items,
                FILM_CUE_REINSTATEMENT,
                paradigm.max_recall,
                CUE_INTERVAL,
                FIRST_CUE_AFTER,
            )
            raw_recalls = np.asarray(recalls).reshape(-1, recalls.shape[-1])
            remapped_recalls = np.asarray(
                standard_remap(
                    raw_recalls,
                    paradigm,
                    show_break=True,
                )
            )
            spc = np.asarray(fixed_pres_spc(remapped_recalls, n_presented), dtype=float)
            pfr = np.asarray(
                fixed_pres_pnr(
                    remapped_recalls,
                    n_presented,
                    query_recall_position=0,
                ),
                dtype=float,
            )
            cue_curves[(float(start_drift_scale), reminder_condition, float(task_scale))] = spc
            pfr_curves[(float(start_drift_scale), reminder_condition, float(task_scale))] = pfr
            totals = phase_recall_totals(spc)
            for phase, total in totals.items():
                cue_rows.append({
                    "start_drift_scale": float(start_drift_scale),
                    "start_drift_label": START_DRIFT_LABELS[float(start_drift_scale)],
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                    "first_cue_after": FIRST_CUE_AFTER,
                    "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                    "phase": phase,
                    "recall_probability_mass": total,
                })
            for position, probability in enumerate(spc, start=1):
                spc_rows.append({
                    "start_drift_scale": float(start_drift_scale),
                    "start_drift_label": START_DRIFT_LABELS[float(start_drift_scale)],
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                    "first_cue_after": FIRST_CUE_AFTER,
                    "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                    "position": position,
                    "phase": position_phase[position - 1],
                    "recall_probability": float(probability),
                })
            for position, probability in enumerate(pfr, start=1):
                pfr_rows.append({
                    "start_drift_scale": float(start_drift_scale),
                    "start_drift_label": START_DRIFT_LABELS[float(start_drift_scale)],
                    "reminder_condition": reminder_condition,
                    "task_condition": TASK_MCF_LABELS[float(task_scale)],
                    "task_mcf_scale": float(task_scale),
                    "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
                    "first_cue_after": FIRST_CUE_AFTER,
                    "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
                    "position": position,
                    "phase": position_phase[position - 1],
                    "first_recall_probability": float(probability),
                })

In [ ]:
phase_totals_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_phase_totals.csv"
diagnostic_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_diagnostic.csv"
spc_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_spc.csv"
pfr_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_pfr.csv"

fieldnames = [
    "start_drift_scale",
    "start_drift_label",
    "reminder_condition",
    "task_condition",
    "task_mcf_scale",
    "film_cue_reinstatement",
    "first_cue_after",
    "film_source_start_drift_rate",
    "phase",
    "recall_probability_mass",
]
write_rows(phase_totals_path, fieldnames, cue_rows)
write_rows(diagnostic_path, fieldnames, cue_rows)
write_rows(
    spc_path,
    [
        "start_drift_scale",
        "start_drift_label",
        "reminder_condition",
        "task_condition",
        "task_mcf_scale",
        "film_cue_reinstatement",
        "first_cue_after",
        "film_source_start_drift_rate",
        "position",
        "phase",
        "recall_probability",
    ],
    spc_rows,
)
write_rows(
    pfr_path,
    [
        "start_drift_scale",
        "start_drift_label",
        "reminder_condition",
        "task_condition",
        "task_mcf_scale",
        "film_cue_reinstatement",
        "first_cue_after",
        "film_source_start_drift_rate",
        "position",
        "phase",
        "first_recall_probability",
    ],
    pfr_rows,
)


def cue_total(start_drift_scale, reminder_condition, task_scale, phase):
    for row in cue_rows:
        if (
            np.isclose(row["start_drift_scale"], start_drift_scale)
            and row["reminder_condition"] == reminder_condition
            and np.isclose(row["task_mcf_scale"], task_scale)
            and row["phase"] == phase
        ):
            return row["recall_probability_mass"]
    return np.nan


def choose_strong_start_drift():
    baseline = cue_total(0.0, "With reminder", 2.0, "film")
    candidate_summaries = []
    selected = float(START_DRIFT_CANDIDATES[-1])
    for value in START_DRIFT_CANDIDATES:
        strong_reminder = cue_total(value, "With reminder", 2.0, "film")
        weak_reminder = cue_total(value, "With reminder", 1.0, "film")
        max_film = max(
            cue_total(value, reminder_condition, float(task_scale), "film")
            for reminder_condition in REMINDER_CONDITIONS
            for task_scale in task_mcf_values
        )
        improvement = strong_reminder - baseline
        contrast = weak_reminder - strong_reminder
        candidate_summaries.append({
            "start_drift_scale": float(value),
            "strong_reminder_film": strong_reminder,
            "weak_reminder_film": weak_reminder,
            "max_film": max_film,
            "improvement_vs_no_start_drift": improvement,
            "weak_strong_contrast_with_reminder": contrast,
        })
        if selected == float(START_DRIFT_CANDIDATES[-1]) and improvement >= 0.5 and max_film < paradigm.n_film * 0.75:
            selected = float(value)
    return selected, candidate_summaries


SELECTED_STRONG_START_DRIFT, start_drift_candidate_summaries = choose_strong_start_drift()
FINAL_START_DRIFT_VALUES = [
    float(START_DRIFT_BASE_VALUES[0]),
    float(START_DRIFT_BASE_VALUES[1]),
    SELECTED_STRONG_START_DRIFT,
]

summary_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_candidate_summary.csv"
write_rows(
    summary_path,
    [
        "start_drift_scale",
        "strong_reminder_film",
        "weak_reminder_film",
        "max_film",
        "improvement_vs_no_start_drift",
        "weak_strong_contrast_with_reminder",
    ],
    start_drift_candidate_summaries,
)

Markdown(
    f"Selected stronger start drift: {SELECTED_STRONG_START_DRIFT:.1f}. "
    f"Saved phase totals to `{phase_totals_path}` and candidate summary to `{summary_path}`."
)

In [ ]:
PANEL_LETTER_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
STRUCTURAL_FONTSIZE = 11
SUPPORT_FONTSIZE = 10

BAR_REMINDER_LABELS = {
    "No reminder": "No reminder",
    "With reminder": "Film reminder",
}
REMINDER_COLORS = {
    "No reminder": "#9CA3AF",
    "With reminder": PHASE_COLORS["reminder"],
}


def add_panel_label(axis, label, x=-0.16, y=1.12):
    axis.text(
        x,
        y,
        label,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_FONTSIZE,
        fontweight="bold",
        va="top",
        ha="left",
    )


def plot_2x2_film_recall(axis, start_drift_scale, title=None, show_legend=False, y_upper=None):
    x = np.arange(len(task_mcf_values))
    width = 0.34
    offsets = [-width / 2, width / 2]
    all_values = []
    for offset, reminder_condition in zip(offsets, REMINDER_CONDITIONS):
        values = [
            cue_total(
                start_drift_scale,
                reminder_condition,
                float(task_scale),
                "film",
            )
            for task_scale in task_mcf_values
        ]
        all_values.extend(values)
        axis.bar(
            x + offset,
            values,
            width=width,
            color=REMINDER_COLORS[reminder_condition],
            edgecolor="white",
            linewidth=0.6,
            label=BAR_REMINDER_LABELS[reminder_condition],
            zorder=2,
        )
    if title is not None:
        axis.set_title(title, fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    axis.set_xticks(x)
    axis.set_xticklabels([TASK_MCF_LABELS[float(value)] for value in task_mcf_values])
    axis.set_xlabel("Task encoding", fontsize=STRUCTURAL_FONTSIZE)
    axis.set_ylim(0, y_upper)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#E7EBF0", linewidth=0.45, alpha=0.45, zorder=0)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if show_legend:
        axis.legend(
            frameon=False,
            fontsize=SUPPORT_FONTSIZE,
            loc="upper right",
            borderaxespad=0.3,
        )

In [ ]:
film_values = [
    cue_total(value, reminder_condition, float(task_scale), "film")
    for value in FINAL_START_DRIFT_VALUES
    for reminder_condition in REMINDER_CONDITIONS
    for task_scale in task_mcf_values
]
y_upper = max(1.0, np.ceil(max(film_values) * 1.14 * 4) / 4)

fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.7), sharey=True)
for axis, label, start_drift_scale in zip(axes, ["A", "B", "C"], FINAL_START_DRIFT_VALUES):
    plot_2x2_film_recall(
        axis,
        start_drift_scale,
        title=START_DRIFT_LABELS[float(start_drift_scale)],
        show_legend=label == "C",
        y_upper=y_upper,
    )
    add_panel_label(axis, label)
axes[0].set_ylabel("Film items recalled", fontsize=STRUCTURAL_FONTSIZE)

save_figure(
    str(figure_dir) if figure_dir is not None else "",
    FIGURE_STR,
    "2x2_film_recall_by_start_drift_cue_090_firstcue0",
)

In [ ]:
FOCUS_REMINDER_CONDITION = "With reminder"
FOCUS_TASK_SCALE = 2.0

support_rows = []
support_curves = []
focus_pfr_curves = []
focus_spc_curves = []
for start_drift_scale in FINAL_START_DRIFT_VALUES:
    support, support_share = support_after_initial_film_cue(
        start_drift_scale,
        FOCUS_REMINDER_CONDITION,
        FOCUS_TASK_SCALE,
    )
    support_curves.append(support)
    focus_pfr_curves.append(
        pfr_curves[(float(start_drift_scale), FOCUS_REMINDER_CONDITION, FOCUS_TASK_SCALE)]
    )
    focus_spc_curves.append(
        cue_curves[(float(start_drift_scale), FOCUS_REMINDER_CONDITION, FOCUS_TASK_SCALE)]
    )
    for position, phase in enumerate(position_phase, start=1):
        support_rows.append({
            "start_drift_scale": float(start_drift_scale),
            "start_drift_label": START_DRIFT_LABELS[float(start_drift_scale)],
            "reminder_condition": FOCUS_REMINDER_CONDITION,
            "task_condition": TASK_MCF_LABELS[FOCUS_TASK_SCALE],
            "task_mcf_scale": FOCUS_TASK_SCALE,
            "film_cue_reinstatement": FILM_CUE_REINSTATEMENT,
            "first_cue_after": FIRST_CUE_AFTER,
            "film_source_start_drift_rate": float(PRIMARY_SOURCE_ORIENTATION),
            "position": position,
            "phase": phase,
            "raw_support": float(support[position - 1]),
            "support_share": float(support_share[position - 1]),
        })

support_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_support.csv"
write_rows(
    support_path,
    [
        "start_drift_scale",
        "start_drift_label",
        "reminder_condition",
        "task_condition",
        "task_mcf_scale",
        "film_cue_reinstatement",
        "first_cue_after",
        "film_source_start_drift_rate",
        "position",
        "phase",
        "raw_support",
        "support_share",
    ],
    support_rows,
)

TASK_COLORS = {
    "Weak encoding": "#86B6E2",
    "Strong encoding": "#08306B",
}


def add_phase_boundaries(axis):
    boundaries = np.cumsum([
        paradigm.n_film,
        paradigm.n_break,
        paradigm.n_interference,
    ]) + 0.5
    for boundary in boundaries:
        axis.axvline(boundary, color="#7C8794", linewidth=0.7, alpha=0.45, zorder=1)


def add_reminder_marker(axis, show_label=False):
    reminder_x = paradigm.n_film + paradigm.n_break + 0.5
    axis.axvline(
        reminder_x,
        color=PHASE_COLORS["reminder"],
        linewidth=1.8,
        alpha=0.85,
        zorder=2,
    )
    if show_label:
        axis.text(
            reminder_x - 1.05,
            0.62,
            "Film reminder",
            ha="center",
            va="center",
            fontsize=STRUCTURAL_FONTSIZE,
            fontstyle="italic",
            color=PHASE_COLORS["reminder"],
            rotation=90,
            rotation_mode="anchor",
            transform=axis.get_xaxis_transform(),
            clip_on=False,
            zorder=3,
        )


def style_position_axis(axis, ylabel, title, show_reminder_label=False):
    add_phase_bands(axis, position_phase, fontsize=STRUCTURAL_FONTSIZE)
    add_phase_boundaries(axis)
    add_reminder_marker(axis, show_label=show_reminder_label)
    axis.set_title(title, fontsize=PANEL_TITLE_FONTSIZE, pad=36)
    axis.set_xlabel("Encoded position", fontsize=STRUCTURAL_FONTSIZE)
    axis.set_ylabel(ylabel, fontsize=STRUCTURAL_FONTSIZE)
    axis.set_xlim(1, n_presented)
    axis.tick_params(labelsize=SUPPORT_FONTSIZE)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)


def plot_position_curves(axis, curves, ylabel, title, ylim, show_reminder_label=False):
    style_position_axis(axis, ylabel, title, show_reminder_label=show_reminder_label)
    colors = light_to_dark_colors(len(FINAL_START_DRIFT_VALUES))
    positions = np.arange(1, n_presented + 1)
    for color, value, curve in zip(colors, FINAL_START_DRIFT_VALUES, curves):
        axis.plot(
            positions,
            curve,
            color=color,
            linewidth=1.4,
            label=f"{value:.1f}",
        )
    axis.set_ylim(*ylim)


fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(
    2,
    4,
    height_ratios=[1.05, 1.0],
    hspace=0.55,
    wspace=0.80,
)

axis_a = fig.add_subplot(gs[0, 0:2])
axis_b = fig.add_subplot(gs[0, 2:4], sharex=axis_a)
axis_c = fig.add_subplot(gs[1, 1:3], sharex=axis_a)
legend_axis = fig.add_subplot(gs[1, 3])
legend_axis.axis("off")

support_upper = max(float(np.max(curve)) for curve in support_curves) * 1.08
pfr_upper = max(float(np.max(curve)) for curve in focus_pfr_curves) * 1.14
spc_upper = max(float(np.max(curve)) for curve in focus_spc_curves) * 1.14

plot_position_curves(
    axis_a,
    support_curves,
    "Pre-recall item support",
    "Pre-recall support",
    (-0.02, support_upper),
    show_reminder_label=True,
)
plot_position_curves(
    axis_b,
    focus_pfr_curves,
    "Probability of first recall",
    "Recall initiation",
    (-0.01, pfr_upper),
    show_reminder_label=True,
)
plot_position_curves(
    axis_c,
    focus_spc_curves,
    "Recall probability",
    "Overall recall",
    (-0.025, spc_upper),
    show_reminder_label=True,
)

add_panel_label(axis_a, "A")
add_panel_label(axis_b, "B")
add_panel_label(axis_c, "C")

handles, labels = axis_a.get_legend_handles_labels()
legend = legend_axis.legend(
    handles,
    labels,
    title="Start drift",
    loc="upper left",
    fontsize=SUPPORT_FONTSIZE,
    title_fontsize=SUPPORT_FONTSIZE,
)
legend.get_title().set_multialignment("center")
legend.get_title().set_ha("center")

save_figure(
    str(figure_dir) if figure_dir is not None else "",
    FIGURE_STR,
    "start_drift_mechanism_with_reminder_strong_task_cue_090_firstcue0",
)

In [ ]:
import textwrap

caption_title = "Figure 2 candidate. Start-of-list context reinstatement changes the reminder-by-task encoding pattern."
caption_body = (
    "Bars show expected film items recalled after a supplied film cue before recall begins. "
    "Panels vary start drift while keeping the film cue, task-encoding manipulation, and film-source orientation fixed. "
    "Higher start drift increases the influence of the start-of-list context at retrieval onset, testing whether the film-reminder loss remains when retrieval is deliberately started from an early-film state."
)

image_path = project_root / "figures" / "simulation1_start_drift_sweep_cued_2x2_film_recall_by_start_drift_cue_090_firstcue0.png"
image = plt.imread(image_path)
fig, axis = plt.subplots(figsize=(10.8, 5.1))
axis.imshow(image)
axis.axis("off")
fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.27)
fig.text(
    0.045,
    0.175,
    caption_title,
    fontsize=11,
    fontweight="bold",
    ha="left",
    va="bottom",
)
fig.text(
    0.045,
    0.045,
    textwrap.fill(caption_body, width=155),
    fontsize=10,
    ha="left",
    va="bottom",
)

caption_base = project_root / "figures" / "simulation1_start_drift_sweep_cued_2x2_film_recall_by_start_drift_cue_090_firstcue0_with_caption"
fig.savefig(f"{caption_base}.png", bbox_inches="tight", dpi=600)
fig.savefig(f"{caption_base}.svg", bbox_inches="tight")
fig.savefig(f"{caption_base}.pdf", bbox_inches="tight")
plt.close(fig)

In [ ]:
print("Start-drift sweep, film cue before first recall")
for start_drift_scale in FINAL_START_DRIFT_VALUES:
    print(f"  {START_DRIFT_LABELS[float(start_drift_scale)]}")
    for reminder_condition in REMINDER_CONDITIONS:
        for task_scale in task_mcf_values:
            film_total = cue_total(start_drift_scale, reminder_condition, float(task_scale), "film")
            task_total = cue_total(start_drift_scale, reminder_condition, float(task_scale), "task")
            print(
                f"    {reminder_condition}, {TASK_MCF_LABELS[float(task_scale)]}: "
                f"film={film_total:.3f}, task={task_total:.3f}"
            )
print("Candidate diagnostics")
for row in start_drift_candidate_summaries:
    print(
        f"  {row['start_drift_scale']:.1f}: "
        f"strong reminder film={row['strong_reminder_film']:.3f}, "
        f"improvement={row['improvement_vs_no_start_drift']:.3f}, "
        f"max film={row['max_film']:.3f}"
    )